In [91]:
import numpy as np
import pandas as pd

In [92]:
df=pd.read_csv('insurance.csv')

In [93]:
df

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520
...,...,...,...,...,...,...,...
1333,50,male,30.970,3,no,northwest,10600.54830
1334,18,female,31.920,0,no,northeast,2205.98080
1335,18,female,36.850,0,no,southeast,1629.83350
1336,21,female,25.800,0,no,southwest,2007.94500


In [94]:
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder

In [95]:
categorical_cols = ["sex", "smoker", "region"]

In [96]:
encoder = OneHotEncoder(drop='first', sparse_output=False)
encoded = encoder.fit_transform(df[categorical_cols])
encoded_df = pd.DataFrame(encoded, columns=encoder.get_feature_names_out())
df = df.drop(columns=categorical_cols)
df = pd.concat([df, encoded_df], axis=1)

In [97]:
df

,age,bmi,children,charges,sex_male,smoker_yes,region_northwest,region_southeast,region_southwest
0,19,27.900,0,16884.92400,0.0,1.0,0.0,0.0,1.0
1,18,33.770,1,1725.55230,1.0,0.0,0.0,1.0,0.0
2,28,33.000,3,4449.46200,1.0,0.0,0.0,1.0,0.0
3,33,22.705,0,21984.47061,1.0,0.0,1.0,0.0,0.0
4,32,28.880,0,3866.85520,1.0,0.0,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...
1333,50,30.970,3,10600.54830,1.0,0.0,1.0,0.0,0.0
1334,18,31.920,0,2205.98080,0.0,0.0,0.0,0.0,0.0
1335,18,36.850,0,1629.83350,0.0,0.0,0.0,1.0,0.0
1336,21,25.800,0,2007.94500,0.0,0.0,0.0,0.0,1.0


In [98]:
X = df.drop("charges", axis=1)
y = df["charges"]

In [99]:

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

In [100]:
from sklearn.linear_model import LinearRegression

In [101]:
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)

In [102]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
def evaluate(y_test, y_pred):
    print("MAE:", mean_absolute_error(y_test, y_pred))
    print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
    print("R2 Score:", r2_score(y_test, y_pred))


In [103]:
print("Linear Regression Performance:")
evaluate(y_test, y_pred_lr)

Linear Regression Performance:
MAE: 4181.194473753654
RMSE: 5796.2846592762735
R2 Score: 0.7835929767120723


In [104]:
from sklearn.ensemble import RandomForestRegressor

In [105]:
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

In [106]:
print("Random Forest Performance:")
evaluate(y_test, y_pred_rf)

Random Forest Performance:
MAE: 2550.0784706115096
RMSE: 4576.299916157115
R2 Score: 0.8651034329144947


In [107]:
from sklearn.linear_model import SGDRegressor

sgd = SGDRegressor(max_iter=1000, tol=1e-3, random_state=42)
sgd.fit(X_train, y_train)


from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

sgd.fit(X_train_scaled, y_train)
y_pred_sgd = sgd.predict(X_test_scaled)

In [108]:
print("SGD Regressor:")
evaluate(y_test, y_pred_sgd)

SGD Regressor:
MAE: 4172.482006957033
RMSE: 5797.322133277084
R2 Score: 0.7835155006160854


In [109]:
from sklearn.tree import DecisionTreeRegressor

dt = DecisionTreeRegressor(max_depth=5, random_state=42)
dt.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)

In [110]:
print("Decision Tree:")
evaluate(y_test, y_pred_dt)

Decision Tree:
MAE: 2930.7675712410432
RMSE: 5082.505543514725
R2 Score: 0.8336098314514943


In [111]:
from sklearn.ensemble import GradientBoostingRegressor
gbr = GradientBoostingRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

gbr.fit(X_train, y_train)
y_pred_gbr = gbr.predict(X_test)

In [112]:
print("Gradient Boosting Regressor:")
evaluate(y_test, y_pred_gbr)

Gradient Boosting Regressor:
MAE: 2443.483262376879
RMSE: 4329.570010504765
R2 Score: 0.8792571359795264


In [113]:
from xgboost import XGBRegressor
xgb = XGBRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

xgb.fit(X_train, y_train)
y_pred_xgb = xgb.predict(X_test)

In [114]:
print("XGBoost Regressor:")
evaluate(y_test, y_pred_xgb)

XGBoost Regressor:
MAE: 2411.127336192484
RMSE: 4320.1596525441655
R2 Score: 0.8797814368972187


In [115]:
from sklearn.model_selection import RandomizedSearchCV
lr_params = {
    "fit_intercept": [True, False]
}
rf_params = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 5, 10],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2]
}
dt_params = {
    "max_depth": [None, 5, 10],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}
sgd_params = {
    "penalty": ["l2", "l1"],
    "alpha": [0.0001, 0.001],
    "learning_rate": ["constant", "optimal"]
}
gbr_params = {
    "n_estimators": [100, 200],
    "learning_rate": [0.01, 0.05, 0.1],
    "max_depth": [3, 4, 5]
}
xgb_params = {
    "n_estimators": [100, 200],
    "learning_rate": [0.01, 0.05],
    "max_depth": [3, 4, 5],
    "subsample": [0.8, 1]
}

In [116]:
from sklearn.linear_model import LinearRegression, SGDRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor

models = {
    "Linear Regression": (LinearRegression(), lr_params),
    "Random Forest": (RandomForestRegressor(random_state=42), rf_params),
    "Decision Tree": (DecisionTreeRegressor(random_state=42), dt_params),
    "Gradient Boosting": (GradientBoostingRegressor(random_state=42), gbr_params),
    "XGBoost": (XGBRegressor(random_state=42), xgb_params),
}

In [117]:
results = {}

for name, (model, params) in models.items():
    print(f"Running Random Search for {name}...")
    
    random_search = RandomizedSearchCV(
        estimator=model,
        param_distributions=params,
        n_iter=10,
        cv=5,
        scoring="r2",
        n_jobs=-1,
        random_state=42
    )
    
    random_search.fit(X_train, y_train)
    
    results[name] = random_search.best_score_
    
    print(f"{name} Best R2: {random_search.best_score_}")
    print("-"*50)
print(f"SGD Best R2: {r2_score(y_test,y_pred_sgd)}")
    

Running Random Search for Linear Regression...
Linear Regression Best R2: 0.7331101109097585
--------------------------------------------------
Running Random Search for Random Forest...


C:\Users\mamta\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\model_selection\_search.py:317: UserWarning: The total space of parameters 2 is smaller than n_iter=10. Running 2 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Random Forest Best R2: 0.8448743018478668
--------------------------------------------------
Running Random Search for Decision Tree...
Decision Tree Best R2: 0.8333512118040565
--------------------------------------------------
Running Random Search for Gradient Boosting...
Gradient Boosting Best R2: 0.839364827258321
--------------------------------------------------
Running Random Search for XGBoost...
XGBoost Best R2: 0.8484090706876408
--------------------------------------------------
SGD Best R2: 0.7835155006160854


In [118]:
print("FINAL MODEL COMPARISON:")
for model, score in results.items():
    print(f"{model}: {score}")

FINAL MODEL COMPARISON:
Linear Regression: 0.7331101109097585
Random Forest: 0.8448743018478668
Decision Tree: 0.8333512118040565
Gradient Boosting: 0.839364827258321
XGBoost: 0.8484090706876408


In [119]:
best_model = max(results, key=results.get)
print(f"Best Model: {best_model}")

Best Model: XGBoost


XGBoost Regressor is the best model(r2 score: 0.87).

As the data is not linearly classified hence Linear regression(0.78) and SGD regressor(0.78) are not performing good.

Gradient Boost's(0.87) performance is comparable to XGBoost but XGBoost has an edge because of its optimization and built in regulariztion term.

Random Forest(0.86) is also performing good but as it is using bagging ensemble it relies on only the best performing descision tree.

Descision tree(0.83) is not performing well because it may be overfitting on the training data.